# 03_limpieza_datos

Proyecto ARIMA / ARIMAX
Modelación epidemiológica con variables meteorológicas.

In [76]:
# Inicio del análisis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  

# Importar datos
ubicacion_janis = r""
ubicacion_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\6_datos_arima\datos_epidemiologicos_2021-2024.xlsx"
df_epi=pd.read_excel(ubicacion_marco)
df_epi.head(1) 



,cod_eve,fec_not,semana,año,cod_pre,cod_sub,edad_,uni_med_,nacionali_,nombre_nacionalidad,...,nom_upgd,npais_proce,ndep_proce,nmun_proce,npais_resi,ndep_resi,nmun_resi,ndep_notif,nmun_notif,nreg
0,210,2021-01-11,1,2021,515402201,1,40,1,170,COLOMBIA,...,ESE HOSPITAL CESAR URIBE PIEDRAHITA,COLOMBIA,ANTIOQUIA,EL BAGRE,COLOMBIA,ANTIOQUIA,EL BAGRE,ANTIOQUIA,CAUCASIA,1800


In [77]:
df_epi.columns 

Index(['cod_eve', 'fec_not', 'semana', 'año', 'cod_pre', 'cod_sub', 'edad_',
       'uni_med_', 'nacionali_', 'nombre_nacionalidad',
       ...
       'nom_upgd', 'npais_proce', 'ndep_proce', 'nmun_proce', 'npais_resi',
       'ndep_resi', 'nmun_resi', 'ndep_notif', 'nmun_notif', 'nreg'],
      dtype='str', length=120)

# Remuestreo de los datos meteorológicos a frecuencia semanal 

In [78]:
import pandas as pd

# Asegurar formato datetime
df_epi["fec_not"] = pd.to_datetime(df_epi["fec_not"])

# Ordenar cronológicamente
df_epi = df_epi.sort_values("fec_not").reset_index(drop=True)

# Crear columnas de año y semana epidemiológica (ISO)
df_epi["año"] = df_epi["fec_not"].dt.isocalendar().year
df_epi["semana_epidemiologica"] = df_epi["fec_not"].dt.isocalendar().week

# Agrupar casos por año y semana
df_casos_semana = (
    df_epi
    .groupby(["año", "semana_epidemiologica"])
    .size()
    .reset_index(name="casos_dengue")
)

# Crear fecha base usando ISO (lunes de la semana)
df_casos_semana["fecha"] = pd.to_datetime(
    df_casos_semana["año"].astype(str)
    + df_casos_semana["semana_epidemiologica"].astype(str)
    + "1",
    format="%G%V%u"
)

# Ajustar para que la semana empiece en sábado
df_casos_semana["fecha"] = df_casos_semana["fecha"] - pd.Timedelta(days=2)

# Ordenar columnas
columnas = ["fecha", "año", "semana_epidemiologica", "casos_dengue"]
df_casos_semana = df_casos_semana[columnas]

# Convertir fecha en índice temporal
df_casos_semana = df_casos_semana.set_index("fecha").sort_index()

df_casos_semana.head()

,año,semana_epidemiologica,casos_dengue
fecha,,,
2021-01-09,2021,2,1
2021-01-30,2021,5,1
2021-02-06,2021,6,1
2021-02-13,2021,7,2
2021-02-20,2021,8,1


In [79]:
df_casos_semana.columns 

Index(['año', 'semana_epidemiologica', 'casos_dengue'], dtype='str')

# Obtener los datos meteorológicos 

In [80]:
# Importar datos
ubicacion_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Datos meteorológicos\NASA POWER\Datos_NS_2021-2024.csv"
ubicacion_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\6_datos_arima\datos_meteorologicos_2021-2024.csv"
df_meteo=pd.read_csv(ubicacion_marco, sep = ';')
df_meteo.head(1) 

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,QV2M,RH2M,PRECTOTCORR,WS2M,WS2M_MAX,WS2M_MIN,UV
0,2021,1,28.38,34.96,23.58,16.62,71.66,3.59,0.14,0.24,0.02,2.48


# Renombramiento de los atributos 

In [81]:
# Renombrar columnas del dataframe nasa
df_meteo.rename(columns={
    'YEAR': 'año',
    'DOY': 'dia',
    'T2M': 'temp',
    'T2M_MAX': 'temp_max',
    'T2M_MIN': 'temp_min',
    'QV2M': 'hum_esp',
    'RH2M': 'hum_rel',
    'PRECTOTCORR': 'prec',
    'WS2M': 'vel_vi',
    'WS2M_MAX': 'vel_vi_max',
    'WS2M_MIN': 'vel_vi_min',
    'UV': 'uv'
}, inplace=True)
df_meteo.head(1)

,año,dia,temp,temp_max,temp_min,hum_esp,hum_rel,prec,vel_vi,vel_vi_max,vel_vi_min,uv
0,2021,1,28.38,34.96,23.58,16.62,71.66,3.59,0.14,0.24,0.02,2.48


# 5. Ordenar los datos meteorológicos cronológicamente en formato DD-MM-AAAA

In [82]:
# Asegurar que las columnas 'año' y 'dia' sean numéricas 
df_meteo["año"] = df_meteo["año"].astype(int)
df_meteo["dia"] = df_meteo["dia"].astype(int)

# Crear la columna 'fecha' combinando año y día del año 
df_meteo["fecha"] = pd.to_datetime(df_meteo["año"].astype(str), format="%Y") + pd.to_timedelta(df_meteo["dia"] - 1, unit="D")

# Eliminar las columnas originales 'año' y 'dia'
df_meteo = df_meteo.drop(columns=[ "dia"])

# Reordenar las columnas para que 'fecha' quede al inicio 
cols = ["fecha"] + [c for c in df_meteo.columns if c != "fecha"]
df_meteo = df_meteo[cols]

# Ordenar cronológicamente 
df_meteo_crono = df_meteo.sort_values("fecha").reset_index(drop=True)

df_meteo_crono.head()


,fecha,año,temp,temp_max,temp_min,hum_esp,hum_rel,prec,vel_vi,vel_vi_max,vel_vi_min,uv
0,2021-01-01,2021,28.38,34.96,23.58,16.62,71.66,3.59,0.14,0.24,0.02,2.48
1,2021-01-02,2021,27.44,33.39,23.64,17.98,80.54,12.12,0.10,0.23,0.04,2.18
2,2021-01-03,2021,28.64,35.09,23.84,17.89,75.41,4.03,0.14,0.32,0.07,2.48
3,2021-01-04,2021,28.48,35.05,23.69,15.91,68.69,0.70,0.14,0.25,0.07,2.47
4,2021-01-05,2021,27.84,34.96,22.71,15.16,68.47,0.62,0.16,0.34,0.03,2.30


# Remuestrear los datos meteorológicos a frecuencia de semana epidemiológica 

In [83]:
import pandas as pd

# 2. Asegurar que la columna 'fecha' es tipo datetime
df_meteo_crono["fecha"] = pd.to_datetime(df_meteo_crono["fecha"])

# 3. Crear columna binaria: día con lluvia (1 si Prec ≥ 1 mm)
df_meteo_crono["lluvia"] = (df_meteo_crono["prec"] >= 1).astype(int)

# 4. Definir la fecha como índice temporal
df_meteo_crono = df_meteo_crono.set_index("fecha")

# 5. Remuestrear de diario a semanal (domingo a sábado)
df_meteo_crono_semanal = df_meteo_crono.resample("W-SAT").agg({
    "temp": "mean",
    "temp_max": "mean",
    "temp_min": "mean",
    "hum_esp": "mean",
    "hum_rel": "mean",
    "prec": "sum",
    "lluvia": "sum",
    "vel_vi": "mean",
    "vel_vi_max": "mean",
    "vel_vi_min": "mean",
    "uv": "mean"
})

# 6. Renombrar variable
df_meteo_crono_semanal = df_meteo_crono_semanal.rename(columns={"lluvia": "dias_lluvia"})

# 7. Convertir índice en columna
df_meteo_crono_semanal = df_meteo_crono_semanal.reset_index()

# 8. Crear semana epidemiológica
df_meteo_crono_semanal["semana_epidemiologica"] = df_meteo_crono_semanal["fecha"].dt.isocalendar().week

# 9. Crear año
df_meteo_crono_semanal["año"] = df_meteo_crono_semanal["fecha"].dt.year

# 10. Crear nueva fecha basada en (año + semana) donde la semana empieza en sábado
df_meteo_crono_semanal["fecha"] = (
    pd.to_datetime(df_meteo_crono_semanal["año"].astype(str) + "-01-01")
    + pd.to_timedelta((df_meteo_crono_semanal["semana_epidemiologica"] - 1) * 7, unit="D")
)

# Ajustar para que caiga en sábado
df_meteo_crono_semanal["fecha"] = df_meteo_crono_semanal["fecha"] + pd.offsets.Week(weekday=5)

# 11. Ordenar columnas
columnas = ["fecha", "año", "semana_epidemiologica"] + [
    col for col in df_meteo_crono_semanal.columns 
    if col not in ["fecha", "año", "semana_epidemiologica"]
]

df_meteo_crono_semanal = df_meteo_crono_semanal[columnas]

# 12. Dejar fecha como índice
df_meteo_crono_semanal = df_meteo_crono_semanal.set_index("fecha")

df_meteo_crono_semanal.head()

,año,semana_epidemiologica,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv
fecha,,,,,,,,,,,,,
2022-01-01,2021,53,27.910000,34.175000,23.610000,17.300000,76.100000,15.71,2,0.120000,0.235000,0.030000,2.330000
2021-01-02,2021,1,28.252857,34.200000,23.832857,16.308571,70.508571,5.72,1,0.121429,0.244286,0.047143,2.222857
2021-01-09,2021,2,28.687143,34.910000,24.195714,17.318571,72.885714,19.15,5,0.117143,0.208571,0.038571,2.254286
2021-01-16,2021,3,29.592857,36.372857,24.090000,16.112857,65.122857,0.77,0,0.124286,0.225714,0.045714,2.420000
2021-01-23,2021,4,29.190000,35.978571,24.200000,16.511429,68.068571,12.92,4,0.122857,0.220000,0.035714,2.477143


In [84]:
df_meteo_crono_semanal.columns

Index(['año', 'semana_epidemiologica', 'temp', 'temp_max', 'temp_min',
       'hum_esp', 'hum_rel', 'prec', 'dias_lluvia', 'vel_vi', 'vel_vi_max',
       'vel_vi_min', 'uv'],
      dtype='str')

# Unir las bases de datos epidemiológicas con la base de datos meteorológica de acuerdo al criterop año semana epidemiológica 


In [86]:
# Asegurar que ambos dataframes estén indexados por fecha
df_meteo_crono_semanal = df_meteo_crono_semanal.sort_index()
df_casos_semana = df_casos_semana.sort_index()

# Unir por índice (fecha)
df_epi_meteo = df_meteo_crono_semanal.join(df_casos_semana[["casos_dengue"]], how="left")

# Rellenar semanas sin casos
df_epi_meteo["casos_dengue"] = df_epi_meteo["casos_dengue"].fillna(0).astype(int)

# Volver a crear columnas de año y semana desde el índice
df_epi_meteo["año"] = df_epi_meteo.index.isocalendar().year
df_epi_meteo["semana_epidemiologica"] = df_epi_meteo.index.isocalendar().week

# Reordenar columnas
columnas = ["año", "semana_epidemiologica", "casos_dengue"] + [
    c for c in df_epi_meteo.columns 
    if c not in ["año", "semana_epidemiologica", "casos_dengue"]
]

df_epi_meteo = df_epi_meteo[columnas]

df_epi_meteo.head()

,año,semana_epidemiologica,casos_dengue,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv
fecha,,,,,,,,,,,,,,
2021-01-02,2020,53,0,28.252857,34.200000,23.832857,16.308571,70.508571,5.72,1,0.121429,0.244286,0.047143,2.222857
2021-01-09,2021,1,1,28.687143,34.910000,24.195714,17.318571,72.885714,19.15,5,0.117143,0.208571,0.038571,2.254286
2021-01-16,2021,2,0,29.592857,36.372857,24.090000,16.112857,65.122857,0.77,0,0.124286,0.225714,0.045714,2.420000
2021-01-23,2021,3,0,29.190000,35.978571,24.200000,16.511429,68.068571,12.92,4,0.122857,0.220000,0.035714,2.477143
2021-01-30,2021,4,1,29.454286,35.882857,24.684286,17.427143,70.105714,17.94,3,0.120000,0.221429,0.030000,2.290000


In [61]:
df_casos_semana.columns

Index(['año', 'semana_epidemiologica', 'casos_dengue', 'fecha'], dtype='str')

In [49]:
df_meteo_crono_semanal.columns

Index(['fecha', 'semana_epidemiologica', 'temp', 'temp_max', 'temp_min',
       'hum_esp', 'hum_rel', 'prec', 'dias_lluvia', 'vel_vi', 'vel_vi_max',
       'vel_vi_min', 'uv', 'año'],
      dtype='str')

In [87]:
# convertir este dataset fusionado en archivo .csv para que pueda ser utilizado en el análisis de series de tiempo con ARIMA
df_epi_meteo.to_csv("datos_fusionados_semanales.csv", index=True) 